- https://simonxin.com/blogs/llm_reasoning_materials/index.html

In [4]:
from IPython.display import Image

In [2]:
Image(url='./imgs/grpo_highlights.png', width=400)

In [7]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo-1.png', width=500)

In [6]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo-2.png', width=500)

- $\pi_\theta(y_{i,t}|x,y_{i,<t})$ is the **token-wise likelihood** on one training sample $(x,y)$
    - This term is identical to the pre-training loss. The difference is that GRPO involves two additional scaling factors:
        - $\frac{1}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$
        - $\hat{A}_{i,t}$

### token-wise issue??

- $(x, y_i)$ is sampled from $\pi_{\theta_\text{old}}$ but used to update $\pi_\theta$, However, the possible value of $\frac{1}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$ ranges from  1 to $\infty$
- https://arxiv.org/pdf/2505.12929
    - Do Not Let **Low-Probability Tokens** Over-Dominate in RL for LLMs
    - $C_l \cdot |w_{i,t}| \cdot \sqrt{\frac{N}{N-1}} \cdot (1 - \pi_\theta(o_{i,t})) \le ||\delta_l(o_{i,t})|| \le D_l \cdot |w_{i,t}| \cdot \sqrt{2} \cdot (1 - \pi_\theta(o_{i,t}))$
        - 梯度的范数（即梯度的大小）||δl(oi,t)|| 与 (1 - πθ(oi,t)) 这一项成正比。
    - 在计算上，低概率词元可以被看作是那些在强化学习更新步骤中，贡献了绝大部分梯度信号的“离群点（outliers）”。
        - 由于低概率词元的梯度值比高概率词元大几个数量级，当它们被一起平均时，低概率词元的梯度信号会“淹没”高概率词元的梯度信号。

## (sequence) length bias in RL?

> Implementation + data > algorithm

Three different ways to sum loss. (Setup: G is a group of completions per a question, each response has o tokens.)

In [3]:
Image(url='./imgs/length_bias.png', width=400)

### GSPO

- verl/trainer/config/actor/actor.yaml
    - policy_loss.loss_mode: vanilla / clip-cov / kl-cov /gpg

$$
\mathcal{J}_{\text{GRPO}} = \frac{1}{|y_i|}\sum_{t=1}^{|y_i|} \frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}
$$
$$
\begin{aligned}     \mathcal{J}_{\text{GSPO}} &= \left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_\text{old}}(y_i|x)}\right)^{\frac{1}{|y_i|}} \\     & = \left(\frac{\Pi_{t=1}^{|y_i|} \pi_\theta(y_{i,t}|x,y_{i,<t})}{\Pi_{t=1}^{|y_i|} \pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}\right)^{\frac{1}{|y_i|}} \\     & = \exp\left(\frac{1}{|y_i|}\log\left(     \frac{\Pi_{t=1}^{|y_i|} \pi_\theta(y_{i,t}|x,y_{i,<t})}{\Pi_{t=1}^{|y_i|} \pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}     \right)\right) \\     & = \exp\left(\frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \log\left(     \frac{ \pi_\theta(y_{i,t}|x,y_{i,<t})}{ \pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}     \right)\right)     \end{aligned}
$$

- GRPO is doing the arithmetic mean of $\frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$
- GSPO is doing the geometric mean of $\frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$
    - $\sqrt[n]{a_1 a_2 \cdots a_n \vphantom{t}} = \exp \left( \frac{\ln a_1 + \ln a_2 + \cdots + \ln a_n }{n} \right)$
- Compared to arithmetic mean, geometric mean is less sensitive to extreme outliers. To see this, let's consider a simpler example and exercise some high-school math!

In [11]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/gspo_loss.png', width=600)

In [9]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo_gradient.png', width=500)

In [10]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo_gradient.png', width=500)

- In GSPO, all tokens in a sequence share the same scaling factor $\left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_\text{old}}(y_i|x)}\right)^{\frac{1}{|y_i|}}$

#### verl impl (core-algos.py)

$$
s_i(\theta) = \left( \frac{\pi_{\theta}(y_i|x)}{\pi_{\theta_{old}}(y_i|x)} \right)^{\frac{1}{|y_i|}} = \exp\left[ \frac{1}{|y_i|} \sum_t \log\left( \frac{\pi_{\theta}(y_{i,t}|x, y_{i,<t})}{\pi_{\theta_{old}}(y_{i,t}|x, y_{i,<t})} \right) \right]
$$

$$
s_{i,t}(\theta) = \text{sg}[s_i(\theta)] \cdot \frac{\pi_{\theta}(y_{i,t}|x, y_{i,<t})}{\text{sg}[\pi_{\theta}(y_{i,t}|x, y_{i,<t})]}
$$

$$
\log(s_{i,t}(\theta)) = \text{sg}[\log(s_i(\theta))] + \text{log\_prob} - \text{sg}[\text{log\_prob}]
$$

- sg：stop gradient
- GSPO 的核心思想是，对于一个序列 y_i 中的每一个词元 y_{i,t}，它的策略梯度更新都应该被同一个序列级别的因子 s_i(θ) 来缩放。
- 直接在代码中实现上述 J_GSPO(θ) 目标会遇到一个问题：损失函数中的 s_i(θ) 依赖于整个序列的对数概率 log_prob。当自动微分框架（如 PyTorch）计算梯度时，对于序列中的某个特定词元 y_{i,t}，它的梯度会受到序列中所有其他词元 y_{i,j} (j≠t) 的影响，因为它们都出现在 s_i(θ) 的计算中。
    - 如果用这个 $s_{i,t}(θ)$ 来构造一个 PPO 风格的词元级别损失 $L_t = s_{i,t}(\theta) \cdot \hat{A}_i$，它的梯度 ∇_θ L_t 是什么？
        - $∇_θ L_t = (∇_θ s_{i,t}(\theta)) \cdot \hat{A}_i$
        - $∇_θ s_{i,t}(\theta)$。由于 `sg[...]` 部分被视为常数，我们可以把它们记作 $C_1 = sg[s_i(θ)]$ 和 $C_2 = sg[π_θ(y_{i,t}|...)]$。
            - $s_{i,t}(\theta) = \frac{C_1}{C_2} \cdot \pi_{\theta}(y_{i,t}|x, y_{i,<t})$
        - 对上式求导： $∇_θ s_{i,t}(\theta) = \frac{C_1}{C_2} \cdot \nabla_θ \pi_{\theta}(y_{i,t}|x, y_{i,<t})$

```python
# compute sequence-level importance ratio:
# si(θ) = (π_θ(yi|x)/π_θold(yi|x))^(1/|yi|) =
# exp [(1/|y_i|) * Σ_t log(π_θ(y_i,t|x,y_i,<t)/π_θold(y_i,t|x,y_i,<t))]
negative_approx_kl = log_prob - old_log_prob

seq_lengths = torch.sum(response_mask, dim=-1).clamp(min=1)
negative_approx_kl_seq = torch.sum(negative_approx_kl * response_mask, dim=-1) / seq_lengths

# Combined ratio at token level:
# s_i,t(θ) = sg[s_i(θ)] · π_θ(y_i,t|x, y_i,<t) / sg[π_θ(y_i,t|x, y_i,<t)]
# In log space: log(s_i,t(θ)) = sg[log(s_i(θ))] + log_prob - sg[log_prob]
log_seq_importance_ratio = negative_approx_kl_seq.detach().unsqueeze(-1) + log_prob - log_prob.detach()


seq_importance_ratio = torch.exp(log_seq_importance_ratio)
pg_losses1 = -advantages * seq_importance_ratio
pg_losses2 = -advantages * torch.clamp(seq_importance_ratio, 1 - clip_ratio_low, 1 + clip_ratio_high)
pg_losses = torch.maximum(pg_losses1, pg_losses2)

# for GSPO, we need to aggregate the loss at the sequence level (seq-mean-token-mean)
pg_loss = agg_loss(loss_mat=pg_losses, loss_mask=response_mask, loss_agg_mode="seq-mean-token-mean")
```

- https://github.com/huggingface/trl/commit/03034317d0be0c259c315f5ffad71be138c17d2c#diff-964e6fd373aa93037604064cb2b822d7f8e2735e33f791065acf2c4c3552d393

### gradient flow

- $ f=\frac{1}{|y_i|} \sum_t \log\left( \frac{\pi_{\theta}(y_{i,t}|x, y_{i,<t})}{\pi_{\theta_{old}}(y_{i,t}|x, y_{i,<t})}\right)$
    - 这一部分对 $\pi_{\theta}(y_{i,t}|x, y_{i,<t})$ 求导
    - $f\cdot\frac{\nabla\pi_\theta}{\pi_\theta}$

In [15]:
import torch

# --- 设定 (通用部分) ---
# 这些值在两个实验中保持不变
old_log_probs = torch.tensor([0.5, 1.5])
A1, A2 = 2.0, 3.0
initial_w_value = [2.0]

print("--- 实验一：直接实现 GSPO 的梯度分析 ---")

# 为实验一创建独立的计算图
w1 = torch.tensor(initial_w_value, requires_grad=True)
log_probs1 = w1 * torch.tensor([1.0, 2.0])
log_p1_exp1, log_p2_exp1 = log_probs1[0], log_probs1[1]

# 1. 计算序列级别的 log importance ratio
log_s_i_exp1 = 0.5 * torch.sum(log_probs1 - old_log_probs)
s_i_exp1 = torch.exp(log_s_i_exp1)
print(f"序列级别的重要性比率 s_i: {s_i_exp1.item():.2f}")

# 2. 定义损失
loss1_exp1 = s_i_exp1 * A1
loss2_exp1 = s_i_exp1 * A2
total_loss_exp1 = loss1_exp1 + loss2_exp1

# 3. 梯度分析
grad_total_wrt_w1 = torch.autograd.grad(total_loss_exp1, w1, retain_graph=True)[0]
grad_loss1_wrt_w1 = torch.autograd.grad(loss1_exp1, w1, retain_graph=True)[0]
grad_loss2_wrt_w1 = torch.autograd.grad(loss2_exp1, w1, retain_graph=False)[0] # 最后一次，释放图

print(f"总损失对 w 的梯度: {grad_total_wrt_w1.item():.2f}")
print(f"损失1 (loss1) 对 w 的梯度: {grad_loss1_wrt_w1.item():.2f}")
print(f"损失2 (loss2) 对 w 的梯度: {grad_loss2_wrt_w1.item():.2f}")
print(f"梯度之和: {grad_loss1_wrt_w1.item() + grad_loss2_wrt_w1.item():.2f}")
print("\n结论：梯度在 loss1 和 loss2 之间'串扰'。")

--- 实验一：直接实现 GSPO 的梯度分析 ---
序列级别的重要性比率 s_i: 7.39
总损失对 w 的梯度: 55.42
损失1 (loss1) 对 w 的梯度: 22.17
损失2 (loss2) 对 w 的梯度: 33.25
梯度之和: 55.42

结论：梯度在 loss1 和 loss2 之间'串扰'。


In [20]:
import numpy as np
2 * np.exp(0.5 * (3 * 2 - 2)) * 1.5, 3 * np.exp(0.5 * (3 * 2 - 2)) * 1.5

(np.float64(22.16716829679195), np.float64(33.25075244518793))

In [16]:
print("\n\n--- 实验二：使用 sg (detach) 技巧的梯度分析 ---")

# 为实验二创建全新的、独立的计算图
w2 = torch.tensor(initial_w_value, requires_grad=True)
log_probs2 = w2 * torch.tensor([1.0, 2.0])
log_p1_exp2, log_p2_exp2 = log_probs2[0], log_probs2[1]

# 1. 计算序列级别的 log importance ratio
log_s_i_exp2 = 0.5 * torch.sum(log_probs2 - old_log_probs)

# 2. 构造代理的词元级别 log ratio
log_s_i_t_1 = log_s_i_exp2.detach() + log_p1_exp2 - log_p1_exp2.detach()
log_s_i_t_2 = log_s_i_exp2.detach() + log_p2_exp2 - log_p2_exp2.detach()

s_i_t_1 = torch.exp(log_s_i_t_1)
s_i_t_2 = torch.exp(log_s_i_t_2)

# 3. 定义损失
loss1_clever = s_i_t_1 * A1
loss2_clever = s_i_t_2 * A2
total_loss_clever = loss1_clever + loss2_clever

# 4. 梯度分析
grad_total_clever_wrt_w2 = torch.autograd.grad(total_loss_clever, w2, retain_graph=True)[0]
grad_loss1_clever_wrt_w2 = torch.autograd.grad(loss1_clever, w2, retain_graph=True)[0]
grad_loss2_clever_wrt_w2 = torch.autograd.grad(loss2_clever, w2, retain_graph=False)[0] # 最后一次，释放图

print(f"巧妙总损失对 w 的梯度: {grad_total_clever_wrt_w2.item():.2f}")
print(f"巧妙损失1 (loss1_clever) 对 w 的梯度: {grad_loss1_clever_wrt_w2.item():.2f}")
print(f"巧妙损失2 (loss2_clever) 对 w 的梯度: {grad_loss2_clever_wrt_w2.item():.2f}")
print(f"梯度之和: {grad_loss1_clever_wrt_w2.item() + grad_loss2_clever_wrt_w2.item():.2f}")
print("\n结论：梯度被成功解耦，不再'串扰'！")



--- 实验二：使用 sg (detach) 技巧的梯度分析 ---
巧妙总损失对 w 的梯度: 59.11
巧妙损失1 (loss1_clever) 对 w 的梯度: 14.78
巧妙损失2 (loss2_clever) 对 w 的梯度: 44.33
梯度之和: 59.11

结论：梯度被成功解耦，不再'串扰'！


- 对于词元 y_{i,1} (我们的 p1)，它的“局部”策略梯度方向是 ∇_w log_p1。我们用 s_i 和它的优势 A1 来缩放它。 ∇_w J_1 ≈ s_i * A1 * ∇_w log_p1
- 对于词元 y_{i,2} (我们的 p2)，它的“局部”策略梯度方向是 ∇_w log_p2。我们用 s_i 和它的优势 A2 来缩放它。 ∇_w J_2 ≈ s_i * A2 * ∇_w log_p2
- 总梯度是各个“意图梯度”之和: $∇_w J_{total} = ∇_w J_1 + ∇_w J_2 = s_i * A_1 * ∇_w log_{p1} + s_i * A_2 * ∇_w log_{p2}$
    - ∇_w J_{total} = (7.39 * 2.0 * 1.0) + (7.39 * 3.0 * 2.0) = 14.78 + 44.34 = 59.12

In [21]:
7.39 * 2.0 * 1.0, 7.39 * 3.0 * 2.0

(14.78, 44.339999999999996)

$$
∇_θ f(θ) = f(θ) * ∇_θ log(f(θ))
$$